# 02 Ratio BAT State DRLB (Optuna 10)

May 04 DRLB run with fixed linear lambda init and Optuna(10).

May 04 profiles (`may04_*`) set `lambda_min=float('-inf')` and `lambda_max=float('+inf')` so DRLB does not apply finite λ bounds.


In [1]:
import sys
import json
from dataclasses import replace
from pathlib import Path

import pandas as pd

cwd = Path.cwd().resolve()
repo_root = cwd
while repo_root != repo_root.parent and not (repo_root / 'pyproject.toml').exists():
    repo_root = repo_root.parent
if not (repo_root / 'pyproject.toml').exists():
    raise RuntimeError('Could not locate repository root with pyproject.toml')

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from example_notebooks.experiments.drlb.profiles import build_config as build_drlb_config
from example_notebooks.experiments.drlb.profiles import get_profile as get_drlb_profile
from example_notebooks.experiments.shared_runner import run_experiment_inprocess

import importlib
import simulator.model.drlb.state_representations as drlb_state_representations
import simulator.model.drlb.rl_bid_agent_bat as drlb_rl_bid_agent_bat
import simulator.model.drlb_bidder as drlb_bidder_module

importlib.reload(drlb_state_representations)
importlib.reload(drlb_rl_bid_agent_bat)
importlib.reload(drlb_bidder_module)


/Users/amsafin/code/local_ml/rl/bat_venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


<module 'simulator.model.drlb_bidder' from '/Users/amsafin/code/local_ml/rl/bat-autobidding-benchmark/simulator/model/drlb_bidder.py'>

In [2]:
RUN_NAME = 'may04_ratio_bat_state_optuna10'
DRLB_PROFILE = 'may04_ratio_bat_linear_lambda_legacy'
VERBOSE = False
SHOW_PROGRESS = True


In [3]:
config = build_drlb_config(
    run_name=RUN_NAME,
    profile=DRLB_PROFILE,
    split_set='full_train_val_holdout',
)
config = replace(config, n_trials=10)
config = replace(config, refit_on='train_plus_val')
config = replace(config, max_steps=None)

profile_data = get_drlb_profile(DRLB_PROFILE)
base_drlb_params = dict(profile_data['base_drlb_params'])
reference_model_params = dict(profile_data['reference_model_params'])

# Enforce May 04 constraints.
reference_model_params['dqn_gamma'] = 1.0
base_drlb_params['init_lambda'] = 0.0028423174374845716
base_drlb_params['init_lambda_mode'] = 'constant'
base_drlb_params['traffic_path'] = str(repo_root / 'data' / 'traffic_share.csv')

from simulator.model.drlb.state_representations import get_state_repr

state_repr = get_state_repr(profile_data['state_type'])
state_repr.begin_episode(1000.0, total_steps=72)
state_vec_len = len(state_repr.curr_state)
if state_vec_len != state_repr.state_size:
    raise RuntimeError(
        f"State representation mismatch for {profile_data['state_type']}: "
        f"len(curr_state)={state_vec_len}, state_size={state_repr.state_size}. "
        "Rerun from the first cell to refresh imports."
    )

result = run_experiment_inprocess(
    config,
    verbose=VERBOSE,
    show_progress=SHOW_PROGRESS,
    base_drlb_params=base_drlb_params,
    reference_model_params=reference_model_params,
    state_type=profile_data['state_type'],
    objective=profile_data['objective'],
    search_space_fn=profile_data['search_space_fn'],
    n_trials=config.n_trials,
    max_train_steps=config.max_steps,
)

summary = result['summary']
summary_path = config.outputs_dir / 'run_summary.json'
print(f'Run summary: {summary_path}')
print(json.dumps({
    'run_name': config.run_name,
    'profile': DRLB_PROFILE,
    'state_type': profile_data['state_type'],
    'init_lambda': base_drlb_params['init_lambda'],
    'init_lambda_mode': base_drlb_params['init_lambda_mode'],
    'n_trials': config.n_trials,
    'tuning_best_params': summary['tuning']['best_params'],
    'best_val_metrics': summary['tuning']['best_val_metrics'],
    'final_holdout_metrics': summary['final_holdout']['metrics'],
    'diagnostics_png': summary['refit']['combined_diagnostics_plot_path'],
}, indent=2))


autobidder_check campaigns: 100%|██████████| 257/257 [00:11<00:00, 21.62campaign/s, campaign_id=7.46e+7]
[I 2026-05-05 00:08:10,416] A new study created in memory with name: no-name-1ccd8566-85ce-4e15-8228-770091ee6110
Best trial: 0. Best value: 2031.46:  10%|█         | 1/10 [03:26<31:02, 206.90s/it]

[I 2026-05-05 00:11:37,319] Trial 0 finished with value: 2031.4627901173787 and parameters: {'dqn_lr': 0.01, 'reward_net_lr': 0.0003, 'bid_lower_clip': 7, 'bid_upper_clip': 6}. Best is trial 0 with value: 2031.4627901173787.


Best trial: 0. Best value: 2031.46:  20%|██        | 2/10 [07:07<28:40, 215.09s/it]

[I 2026-05-05 00:15:18,135] Trial 1 finished with value: 1745.1029480805646 and parameters: {'dqn_lr': 0.01, 'reward_net_lr': 0.0001, 'bid_lower_clip': 7, 'bid_upper_clip': 4}. Best is trial 0 with value: 2031.4627901173787.


Best trial: 2. Best value: 2407.07:  30%|███       | 3/10 [10:40<24:57, 213.94s/it]

...

[I 2026-05-05 00:38:13,841] Trial 8 finished with value: 1271.5855356768172 and parameters: {'dqn_lr': 0.0001, 'reward_net_lr': 0.0001, 'bid_lower_clip': 2, 'bid_upper_clip': 9}. Best is trial 2 with value: 2407.0699103609472.


Best trial: 2. Best value: 2407.07: 100%|██████████| 10/10 [33:16<00:00, 199.65s/it]


[I 2026-05-05 00:41:26,935] Trial 9 finished with value: 1851.5522495055848 and parameters: {'dqn_lr': 0.001, 'reward_net_lr': 0.01, 'bid_lower_clip': 3, 'bid_upper_clip': 10}. Best is trial 2 with value: 2407.0699103609472.


autobidder_check campaigns: 100%|██████████| 257/257 [00:13<00:00, 18.64campaign/s, campaign_id=7.46e+7]


Run summary: /Users/amsafin/code/local_ml/rl/bat-autobidding-benchmark/example_notebooks/experiments/drlb/may04_ratio_bat_state_optuna10/outputs/run_summary.json
{
  "run_name": "may04_ratio_bat_state_optuna10",
  "profile": "may04_ratio_bat_linear_lambda_legacy",
  "state_type": "ratio_bat",
  "init_lambda": 0.0028423174374845716,
  "init_lambda_mode": "constant",
  "n_trials": 10,
  "tuning_best_params": {
    "dqn_lr": 0.01,
    "reward_net_lr": 0.0003,
    "bid_lower_clip": 1,
    "bid_upper_clip": 9
  },
  "best_val_metrics": {
    "cpc_relative": 470.3492054262821,
    "rmse": 1.6152828032004292,
    "clicks_sum": 2407.0699103609472,
    "quickspend": 0.03501945525291829,
    "skipped_campaigns": 0,
    "time_inference_sec": 17.785976886749268,
    "time_overall_sec": 22.40011191368103,
    "average_end_balance_share": 0.5896788396357265,
    "label": "best_val",
    "train_steps": 48240,
    "last_dqn_loss": 10.776116371154785,
    "last_reward_net_loss": 617.1570434570312,
    

In [4]:
rows = [
    {'artifact': 'run_summary_json', 'path': str(config.outputs_dir / 'run_summary.json')},
    {'artifact': 'metrics_json', 'path': str(config.outputs_dir / 'metrics.json')},
    {'artifact': 'drlb_diagnostics_png', 'path': str(config.outputs_dir / 'drlb_diagnostics.png')},
    {'artifact': 'best_refit_model', 'path': str(config.best_models_dir / 'best_refit.pt')},
]
pd.DataFrame(rows)


,artifact,path
0,run_summary_json,/Users/amsafin/code/local_ml/rl/bat-autobiddin...
1,metrics_json,/Users/amsafin/code/local_ml/rl/bat-autobiddin...
2,drlb_diagnostics_png,/Users/amsafin/code/local_ml/rl/bat-autobiddin...
3,best_refit_model,/Users/amsafin/code/local_ml/rl/bat-autobiddin...
